# Clase 09 — Ejercicio intergrador Clínica MediCare

> Notebook de práctica. Ejecutá las celdas a medida que avanzás en el artículo.

La clínica MediCare lleva varios años registrando turnos médicos en un sistema interno. El sistema fue cambiando de versión con el tiempo, y los datos se fueron exportando a una base de datos SQLite sin demasiado control de calidad.

El director médico necesita un informe con algunas preguntas concretas sobre el funcionamiento de la clínica, pero antes de poder responderlas, alguien tiene que hacer la limpieza. Ese trabajo es el de ustedes, no se les dice qué problemas hay en los datos. Parte del ejercicio es encontrarlos.

> El archivo de la base de datos `medicare.db` ya está disponible para que lo uses en este ejercicio.

### 1) Exploración: ¿qué tiene esta base de datos?

Antes de tocar nada, el primer trabajo es entender qué hay adentro. Conectense a `medicare.db` y respondan estas preguntas.

1. ¿Qué tablas tiene la base de datos?
2. ¿Qué columnas y tipos tiene cada tabla?
3. ¿Cuántas filas tiene cada tabla?

In [29]:
# tu codigo aca
import sqlite3
import pandas as pd

#Conectar python con el archivo
conexion = sqlite3.connect("medicare.db")

#Traer los nombres de las tablas que se encuentran en master
#1)
tablas = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'",conexion)
print(tablas)

#Separar las 3 tablas (cargar la tabla en un DataFrame)

df_medicos = pd.read_sql("SELECT * FROM medicos", conexion)
print(df_medicos)


df_pacientes = pd.read_sql("SELECT * FROM pacientes", conexion)
print(df_pacientes)


df_turnos = pd.read_sql("SELECT * FROM turnos", conexion)
print(df_turnos)

#2) 
df_medicos.info()
df_pacientes.info()
df_turnos.info()

#3)
print(df_medicos.shape)
print(df_pacientes.shape)
print(df_turnos.shape)







        name
0  pacientes
1    medicos
2     turnos
    id               nombre    especialidad matricula activo
0    1    Dra. Beatriz Sosa  Clínica Médica  MP-12345      1
1    2     Dr. Nicolás Vera       Pediatría  MP-23456      1
2    3  Dra. Carmen Aguirre     Cardiología       NaN      1
3    4     Dr. Pablo Méndez  Clínica Médica  MP-45678      0
4    5    Dra. Beatriz Sosa  Clínica Médica  MP-12345      1
5    6   Dr. Sebastián Ríos   Traumatología  MP-56789      1
6    7   Dra. Laura Giménez       Pediatría  MP-67890    NaN
7    8    Dra. Beatriz Sosa      Neurología  MP-21073      0
8    9     Dr. Nicolás Vera  Endocrinología  MP-46953    NaN
9   10  Dra. Carmen Aguirre   Traumatología  MP-15929      0
10  11     Dr. Pablo Méndez     Cardiología  MP-68435      1
11  12   Dr. Sebastián Ríos    Dermatología  MP-53113    NaN
12  13   Dra. Laura Giménez     Ginecología  MP-50573      1
13  14   Dra. Andrea Blanco     Ginecología  MP-53952      0
14  15     Dr. Martín Cufré      

**Documenten sus hallazgos.** Antes de seguir al Paso 2, escriban en comentarios qué encontraron.

### 2) Diagnóstico: ¿qué problemas tienen los datos?

Traigan las tres tablas a DataFrames y hagan un diagnóstico completo de cada una. Para cada tabla respondan:

- ¿Cuántos nulos hay por columna? ¿Qué porcentaje representan?
- ¿Hay filas duplicadas? ¿Cuáles?
- ¿Hay columnas con tipos incorrectos que van a impedir operar con los datos?
- ¿Hay valores que parecen errores aunque no sean nulos? (Pista: revisen la columna `costo` en `turnos`)


In [36]:
# tu codigo aquí
#df_medicos
#df_pacientes
#df_turnos

# 1)Nulos por columna
print(df_medicos.isnull().sum())

#Porcentaje de nulos
porcentaje = (df_medicos.isnull().sum() / len(df_medicos) * 100).round(1)
print(porcentaje)

# 2) Ver si hay filas duplicadas (Las repeticiones son `True`)
print(df_medicos.duplicated().sum())

#Para ver cuáles son esas filas:
print(df_medicos[df_medicos.duplicated()])

#Duplicados viendo por columnas especifica
print(df_medicos.duplicated(subset=["nombre","matricula","activo"]).sum())
#Para que se vea la lista de los dos duplicados
print(df_medicos[df_medicos.duplicated(subset=["nombre","matricula","activo"], keep=False)])


# 3) Tipos incorrectos: Buscá columnas que aparezcan como str (texto) pero que deberían ser números. 

print(df_medicos.dtypes) #te da una lista con cada columna y su tipo.
#df_medicos: la columna activo está como texto y debería ser numérica o booleana.

#Sacando los NAN de activo y matricula y sacando los DUPLICADOS de nombre matricula y activo
df_medicos_limpio = (df_medicos.dropna(subset=["activo", "matricula"]).drop_duplicates(subset=["nombre", "matricula", "activo"]))

#Convertir con map en vez de con astype:
df_medicos_limpio["activo"] = df_medicos_limpio["activo"].map({"1": True, "0": False})
#.map cambia cada dato de la columna por otro que vos elegís- Ej: “1” por True y el “0” por False.

#Uso el map porque el astype(bool) solo mira si el dato está vacío o no, y como “0” es un texto con algo escrito, 
#lo toma como True.

print(df_medicos_limpio)



id              0
nombre          0
especialidad    0
matricula       2
activo          4
dtype: int64
id               0.0
nombre           0.0
especialidad     0.0
matricula       10.0
activo          20.0
dtype: float64
0
Empty DataFrame
Columns: [id, nombre, especialidad, matricula, activo]
Index: []
1
   id             nombre    especialidad matricula activo
0   1  Dra. Beatriz Sosa  Clínica Médica  MP-12345      1
4   5  Dra. Beatriz Sosa  Clínica Médica  MP-12345      1
id              int64
nombre            str
especialidad      str
matricula         str
activo            str
dtype: object
    id               nombre    especialidad matricula  activo
0    1    Dra. Beatriz Sosa  Clínica Médica  MP-12345    True
1    2     Dr. Nicolás Vera       Pediatría  MP-23456    True
3    4     Dr. Pablo Méndez  Clínica Médica  MP-45678   False
5    6   Dr. Sebastián Ríos   Traumatología  MP-56789    True
7    8    Dra. Beatriz Sosa      Neurología  MP-21073   False
9   10  Dra. Carmen Ag

**Antes de seguir al Paso 3, escriban una lista con todos los problemas que encontraron, uno por línea.** No empiecen a limpiar todavía.

### 3) Decisiones: ¿qué hacemos con cada problema?

Este es el paso más importante del ejercicio. Para cada problema que encontraron en el Paso 2, decidan qué van a hacer y por qué. No hay una respuesta única correcta — lo que importa es que la decisión sea coherente con el análisis que necesitan hacer después.

Completen esta tabla antes de escribir código:

| Tabla | Columna | Problema | Decisión | Justificación |
|-------|---------|----------|----------|---------------|
|pacientes| ciudad| 2 nulos  | Pondria "Sin definir"    | no borraria a un paciente solo porque no completo el campo  |
|pacientes| dni   | 1 nulo   | Borraria la fila | Dato importante no ingresado
| ...     | ...   | ...      | ...      | ... |


### 4) Limpieza: aplicar las decisiones

Ahora sí, implementen las decisiones del Paso 3. Recuerden:

- Primero eliminen duplicados
- Después manejen nulos
- Después conviertan tipos

In [92]:
# Limpieza de pacientes
df_pacientes_limpio = df_pacientes.copy()

# Tu código acá
df_pacientes_limpio = (df_pacientes_limpio.drop_duplicates(subset=["nombre", "dni"]))
df_pacientes_limpio = (df_pacientes_limpio.dropna(subset=["dni", "fecha_nacimiento", "ciudad"]))

print(df_pacientes_limpio.head(15))
                       
# Limpieza de médicos

df_medicos_limpio = df_medicos.copy()
# Tu código acá
df_medicos_limpio = (df_medicos_limpio.drop_duplicates(subset=["nombre", "matricula", "activo"]))
df_medicos_limpio = (df_medicos_limpio.dropna(subset=["activo", "matricula"]))
df_medicos_limpio["activo"] = df_medicos_limpio["activo"].map({"1": True, "0": False})

print(df_medicos_limpio.head(15))


# Limpieza de turnos

df_turnos_limpio = df_turnos.copy()
# Tu código acá
df_turnos_limpio = (df_turnos_limpio.drop_duplicates(subset=["medico_id", "fecha", "hora"]))

df_turnos_limpio["costo"] = pd.to_numeric(df_turnos_limpio["costo"], errors="coerce")
df_turnos_limpio = df_turnos_limpio.dropna()

df_turnos_limpio = df_turnos_limpio[df_turnos_limpio["costo"] >= 0]

df_turnos_limpio.head(50)


    id           nombre         dni fecha_nacimiento        ciudad  \
0    1   Martina García  28.341.992       1985-04-12  Buenos Aires   
1    2     Roberto Díaz  31.882.441       1990-11-30       Córdoba   
4    5       Tomás Ruiz  40.223.117       2000-03-15       Rosario   
5    6  Valentina López  38.774.552       1995-09-08       Mendoza   
6    7   Carlos Herrera  25.990.334       1978-01-25  Buenos Aires   
8    9       Diego Soto  42.118.776       2003-12-01       Rosario   
12  13     Sofía Castro  44.001.557       2005-02-28       Mendoza   
14  15    Marcos Acosta  33.558.114       1993-05-10  Buenos Aires   
15  16   Valentina Vera  41.288.257       1966-02-20       Mendoza   
16  17    Pablo Álvarez  22.075.745       1975-01-08  Bahía Blanca   
17  18  Carla Fernández  41.283.226       1962-02-07       Córdoba   
19  20      Rocío Ortiz  36.665.640       1979-02-27       Córdoba   
20  21  Carlos Martínez  36.963.548       1962-12-09       Neuquén   
21  22   Facundo Tor

,id,paciente_id,medico_id,fecha,hora,motivo,costo,estado
0,1,1,1,2024-03-01,09:00,Control anual,3500.0,realizado
1,2,2,2,2024-03-01,10:30,Fiebre,4200.0,realizado
2,3,3,1,2024-03-02,08:30,Dolor abdominal,3500.0,realizado
3,4,4,3,2024-03-02,11:00,Electrocardiograma,8500.0,realizado
6,7,7,1,2024-03-04,09:30,Control anual,3500.0,cancelado
7,8,8,3,2024-03-04,10:00,Consulta cardiológica,8500.0,realizado
8,9,9,7,2024-03-05,11:30,Fiebre,4200.0,realizado
9,10,10,1,2024-03-05,08:00,Chequeo general,3500.0,realizado
11,12,11,4,2024-03-06,15:00,Dolor de espalda,3500.0,realizado
12,13,12,6,2024-03-06,16:30,Esguince tobillo,5500.0,ausente


> **Por qué `.copy()`:** cuando hacen `df_pacientes_limpio = df_pacientes`, no están creando una copia — están apuntando al mismo objeto. Cualquier cambio en `df_pacientes_limpio` modifica también `df_pacientes`. `.copy()` crea una copia independiente, así conservan los datos originales para comparar.

Al terminar, impriman un resumen de qué cambió en cada tabla:

In [82]:
# tu codigo aquí
print("PACIENTES")
print("Filas antes:", len(df_pacientes))
print("Filas después:", len(df_pacientes_limpio))
print("Filas eliminadas:", len(df_pacientes) - len(df_pacientes_limpio))

print("MÉDICOS")
print("Filas antes:", len(df_medicos))
print("Filas después:", len(df_medicos_limpio))
print("Filas eliminadas:", len(df_medicos) - len(df_medicos_limpio))

print("TURNOS")
print("Filas antes:", len(df_turnos))
print("Filas después:", len(df_turnos_limpio))
print("Filas eliminadas:", len(df_turnos) - len(df_turnos_limpio))

PACIENTES
Filas antes: 66
Filas después: 32
Filas eliminadas: 34
MÉDICOS
Filas antes: 20
Filas después: 13
Filas eliminadas: 7
TURNOS
Filas antes: 210
Filas después: 158
Filas eliminadas: 52


In [93]:

df_turnos_limpio["estado"] = df_turnos_limpio["estado"].str.lower()

df_turnos_limpio.to_sql("df_turnos_limpio", conexion, if_exists="replace", index=False)
df_medicos_limpio.to_sql("df_medicos_limpio", conexion, if_exists="replace", index=False)
df_pacientes_limpio.to_sql("df_pacientes_limpio", conexion, if_exists="replace", index=False)

38

### 5) — Análisis: responder las preguntas del director

Con los datos limpios, respondan estas cuatro preguntas. Para cada una, combinen SQL (para traer los datos necesarios) con Pandas (para calcular o presentar el resultado):

**Pregunta 1:** ¿Cuántos turnos realizó cada médico? Mostrá el resultado ordenado de mayor a menor.

**Pregunta 2:** ¿Cuál es el costo total facturado por especialidad médica? Considerá solo los turnos con estado `'realizado'`.

**Pregunta 3:** ¿Qué obra social tiene más pacientes registrados? ¿Y cuántos pacientes no tienen obra social?

**Pregunta 4:** ¿Cuántos turnos fueron cancelados o marcados como `'ausente'`? ¿Qué médico tuvo más cancelaciones?


In [114]:
# 1) Cuantos turnos realizo cada medico

turnosxmedico = pd.read_sql(
    """
    SELECT medico_id,  COUNT(*) AS cantidad
    FROM df_turnos_limpio
    WHERE estado = 'realizado'
    GROUP BY medico_id
    ORDER BY cantidad DESC
    """,
    conexion
)

print(turnosxmedico)



    medico_id  cantidad
0           3        10
1          12         9
2          10         6
3           7         6
4          14         5
5           1         5
6          20         4
7          18         4
8           4         4
9          19         3
10         11         3
11          9         3
12          6         3
13          5         3
14          2         3
15         17         2
16         16         2
17         15         2
18          8         2
19         13         1


In [118]:
# 2)Costo total por especialidad
costoxespecialidad = pd.read_sql(
  """
  SELECT df_medicos_limpio.especialidad, SUM(df_turnos_limpio.costo) AS total
  FROM df_turnos_limpio
  JOIN df_medicos_limpio ON df_turnos_limpio.medico_id = medico_id
  WHERE df_turnos_limpio.estado = 'realizado'
  GROUP BY df_medicos_limpio.especialidad
  """,
 conexion
)

print(costoxespecialidad)


     especialidad      total
0     Cardiología   497900.0
1  Clínica Médica   995800.0
2    Dermatología   995800.0
3     Ginecología   995800.0
4      Neurología   497900.0
5       Pediatría   995800.0
6   Traumatología  1493700.0


In [117]:
#3) cuantos pacientes tiene obra social y cuantos no

pacientesxobrasocial = pd.read_sql(
"""
SELECT obra_social, COUNT(*) AS cantidad
FROM df_pacientes_limpio
GROUP BY obra_social
ORDER BY cantidad DESC
""",
conexion
)

print(pacientesxobrasocial)



     obra_social  cantidad
0  Swiss Medical         9
1         Galeno         6
2            NaN         6
3   Sancor Salud         5
4           PAMI         4
5         Medife         4
6           OSDE         3
7           IOMA         1


In [116]:
#4)Cuantos turnos fureron cancelados o ausentes, que medico tuvo mas cancelaciones:
turnoscancelados = pd.read_sql(
    """
    SELECT medico_id, COUNT (*) AS ausentes
    FROM df_turnos_limpio
    WHERE estado IN ('cancelado', 'ausente')
    GROUP BY medico_id
    ORDER BY ausentes DESC 
    """
    ,
    conexion
)

print(turnoscancelados)

    medico_id  ausentes
0           6         7
1          11         5
2          20         4
3           1         4
4          17         3
5          14         3
6           5         3
7          13         2
8          12         2
9           9         2
10          8         2
11          7         2
12          4         2
13         19         1
14         16         1
15         15         1
16         10         1
17          3         1
18          2         1


### 6) Reporte final

Escriban un bloque de código que imprima un reporte con los resultados de los cuatro puntos anteriores, en formato legible. No es necesario que sea elaborado — lo importante es que quede claro qué encontraron.

In [120]:

print("=" * 50)
print("REPORTE CLÍNICA MEDICARE")
print("=" * 50)

print("=" * 50)
print("Turnos realizados por médico:")
print(turnosxmedico)
print("=" * 50)

print("=" * 50)
print("Costo total facturado por especialidad:")
print(costoxespecialidad)
print("=" * 50)

print("=" * 50)
print("Pacientes por obra social:")
print(pacientesxobrasocial)
print("=" * 50)

print("=" * 50)
print("Turnos cancelados o ausentes:")
print(turnoscancelados)
print("=" * 50)



REPORTE CLÍNICA MEDICARE
Turnos realizados por médico:
    medico_id  cantidad
0           3        10
1          12         9
2          10         6
3           7         6
4          14         5
5           1         5
6          20         4
7          18         4
8           4         4
9          19         3
10         11         3
11          9         3
12          6         3
13          5         3
14          2         3
15         17         2
16         16         2
17         15         2
18          8         2
19         13         1
Costo total facturado por especialidad:
     especialidad      total
0     Cardiología   497900.0
1  Clínica Médica   995800.0
2    Dermatología   995800.0
3     Ginecología   995800.0
4      Neurología   497900.0
5       Pediatría   995800.0
6   Traumatología  1493700.0
Pacientes por obra social:
     obra_social  cantidad
0  Swiss Medical         9
1         Galeno         6
2            NaN         6
3   Sancor Salud         5
4      

→ [Ir a la solución](./solucion.md)